# QuickMapper: from any tabular file to RDF

This notebook is for the case where you have a measurement file but no
existing parser or schema for it. Fill in a short mapping config and you
get a valid RDF graph back within minutes.

**Supported file formats:** CSV, TSV, TXT (auto-sniffed), Excel (.xlsx / .xls), Parquet

**What you provide:**
- Your data file
- A `mapping.yaml` file that names the columns and points each one at an
  ontology class IRI (and an optional unit)

**What you get back:**
- An RDF graph typed as `dcat:Dataset` (or any class you specify)
- One ontology-annotated descriptor node per mapped column
- A pandas DataFrame with the full data, ready to analyse

The result is intentionally lightweight. When the same file type recurs
often enough to warrant a formal schema, the `adding-a-parser.md` guide
explains how to graduate from a mapping config to a proper parser.

In [1]:
%pip install -q semantic-transformers rdflib pyyaml pandas openpyxl pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pathlib, json, yaml, rdflib
from semantic_transformers import QuickMapper

HERE = pathlib.Path().resolve()

## Step 0: Point to your file

Set `data_file` to the path of your measurement file.  The format is
detected automatically from the file extension.

The cell below creates a small example Excel file so the notebook runs
without any external data.  Replace it with your own file path.

In [3]:
# Create a small example file so this notebook is self-contained.
# Replace this with your own file:
#   data_file = pathlib.Path('/path/to/my_measurement.xlsx')

import pandas as pd

_example = pd.DataFrame({
    'Time (s)':        [0.0, 0.5, 1.0, 1.5, 2.0],
    'Force (N)':       [0.0, 120.5, 245.3, 371.8, 498.2],
    'Displacement (mm)': [0.0, 0.05, 0.11, 0.16, 0.21],
    'Temperature (C)': [22.1, 22.2, 22.3, 22.2, 22.1],
})
_example_path = HERE / 'example_measurement.xlsx'
_example.to_excel(_example_path, index=False)

data_file = _example_path
print(f'File: {data_file}')
print(f'Format detected: .{data_file.suffix.lstrip(".")}')

File: /root/semantic-dataspace/semantic-transformers/docs/example_measurement.xlsx
Format detected: .xlsx


## Step 1: Inspect the file

Load the file with pandas first to see what columns it contains.
This helps you fill in the mapping config in the next step.

In [4]:
# Preview the file contents (QuickMapper will re-read it with your mapping applied)
preview = pd.read_excel(data_file) if data_file.suffix.lower() in ('.xlsx', '.xls') \
     else pd.read_csv(data_file)

print(f'{len(preview)} rows x {len(preview.columns)} columns\n')
print('Columns found:')
for col in preview.columns:
    print(f'  {col!r:<30}  dtype: {preview[col].dtype}')

print()
preview.head()

5 rows x 4 columns

Columns found:
  'Time (s)'                      dtype: float64
  'Force (N)'                     dtype: float64
  'Displacement (mm)'             dtype: float64
  'Temperature (C)'               dtype: float64



,Time (s),Force (N),Displacement (mm),Temperature (C)
0,0.0,0.0,0.00,22.1
1,0.5,120.5,0.05,22.2
2,1.0,245.3,0.11,22.3
3,1.5,371.8,0.16,22.2
4,2.0,498.2,0.21,22.1


## Step 2: Write your mapping

Edit the mapping dict below.  For each column you want to semantify:

- `iri`: the ontology class IRI for this measurement type.  If you are
  not sure, a good starting point is to search [BioPortal](https://bioportal.bioontology.org/)
  or [AgroPortal](https://agroportal.lirmm.fr/) for your domain, or use a
  plain `https://example.org/vocab/MyMeasurement` placeholder.
- `unit`: a [QUDT unit IRI](https://qudt.org/vocab/unit/).  Omit if the
  column has no physical unit.

Columns you do not list are still returned in the DataFrame; they just
will not have RDF triples.

The `file:` block is only needed when auto-detection is not enough
(e.g. the file has metadata rows above the header, or a non-standard
separator).

In [5]:
mapping = {
    # Optional: defaults to dcat:Dataset
    'root_type': 'http://www.w3.org/ns/dcat#Dataset',

    # Optional: defaults to the file stem
    'label': 'Example measurement',

    # File reading options (all optional — remove the block if auto-detection works)
    # 'file': {
    #     'skip_rows': 0,
    #     'separator': ',',   # csv/tsv only
    #     'sheet': 0,          # Excel only
    # },

    'columns': {
        'Force (N)': {
            'iri':  'https://w3id.org/pmd/tto/StandardForce',
            'unit': 'http://qudt.org/vocab/unit/N',
        },
        'Displacement (mm)': {
            'iri':  'https://w3id.org/pmd/tto/Extension',
            'unit': 'http://qudt.org/vocab/unit/MilliM',
        },
        'Temperature (C)': {
            'iri':  'https://w3id.org/pmd/co/PMD_0000071',
            'unit': 'http://qudt.org/vocab/unit/DEG_C',
        },
        # 'Time (s)' is left unmapped — it will still appear in the DataFrame
    },
}

# Optionally save it as a YAML file for reuse
mapping_file = HERE / 'my_mapping.yaml'
mapping_file.write_text(yaml.dump(mapping, allow_unicode=True), encoding='utf-8')
print(f'Mapping saved to {mapping_file}')

Mapping saved to /root/semantic-dataspace/semantic-transformers/docs/my_mapping.yaml


## Step 3: Run the mapper

`QuickMapper` reads the file, applies the column annotations, and builds
an RDF graph. No schema files or JSONata transforms are needed.

In [6]:
mapper = QuickMapper(mapping)
result = mapper.run(data_file)

print('Mapping summary:')
print(json.dumps(result.oold_doc, indent=2))

Mapping summary:
{
  "id": "https://example.org/datasets/example_measurement",
  "type": "http://www.w3.org/ns/dcat#Dataset",
  "label": "Example measurement",
  "source": "example_measurement.xlsx",
  "columns": {
    "Force (N)": {
      "iri": "https://w3id.org/pmd/tto/StandardForce",
      "unit": "http://qudt.org/vocab/unit/N"
    },
    "Displacement (mm)": {
      "iri": "https://w3id.org/pmd/tto/Extension",
      "unit": "http://qudt.org/vocab/unit/MilliM"
    },
    "Temperature (C)": {
      "iri": "https://w3id.org/pmd/co/PMD_0000071",
      "unit": "http://qudt.org/vocab/unit/DEG_C"
    }
  }
}


## Step 4: Inspect the time-series data

The full DataFrame is in `result.dataframe`. All columns are present,
including any that were not mapped.

In [7]:
df = result.dataframe
print(f'{len(df)} rows x {len(df.columns)} columns\n')
df

5 rows x 4 columns



,Time (s),Force (N),Displacement (mm),Temperature (C)
0,0.0,0.0,0.00,22.1
1,0.5,120.5,0.05,22.2
2,1.0,245.3,0.11,22.3
3,1.5,371.8,0.16,22.2
4,2.0,498.2,0.21,22.1


In [8]:
# Semantic annotations for the mapped columns
print(f'{"Column":<25}  {"Ontology class":<45}  Unit IRI')
print('-' * 110)
for col in df.columns:
    iri  = result.column_iris.get(col,  '(not mapped)')
    unit = result.column_units.get(col, '')
    cls  = iri.rsplit('/', 1)[-1] if '/' in iri else iri
    print(f'{col:<25}  {cls:<45}  {unit}')

Column                     Ontology class                                 Unit IRI
--------------------------------------------------------------------------------------------------------------
Time (s)                   (not mapped)                                   
Force (N)                  StandardForce                                  http://qudt.org/vocab/unit/N
Displacement (mm)          Extension                                      http://qudt.org/vocab/unit/MilliM
Temperature (C)            PMD_0000071                                    http://qudt.org/vocab/unit/DEG_C


## Step 5: Inspect the RDF graph

The graph contains one `dcat:Dataset` root node and one typed descriptor
node per mapped column.

In [9]:
flat = rdflib.Graph()
for s, p, o, _ in result.graph.quads():
    flat.add((s, p, o))

print(f'Graph contains {len(flat)} triples.\n')
print(flat.serialize(format='turtle'))

Graph contains 16 triples.

@prefix dcat: <http://www.w3.org/ns/dcat#> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix ns1: <http://qudt.org/schema/qudt/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<https://example.org/datasets/example_measurement> a dcat:Dataset ;
    rdfs:label "Example measurement" ;
    dcterms:source "example_measurement.xlsx" ;
    dcterms:title "Example measurement" ;
    dcat:distribution <https://example.org/datasets/example_measurement/Displacement_(mm)>,
        <https://example.org/datasets/example_measurement/Force_(N)>,
        <https://example.org/datasets/example_measurement/Temperature_(C)> .

<https://example.org/datasets/example_measurement/Displacement_(mm)> a <https://w3id.org/pmd/tto/Extension> ;
    rdfs:label "Displacement (mm)" ;
    ns1:hasUnit <http://qudt.org/vocab/unit/MilliM> .

<https://example.org/datasets/example_measurement/Force_(N)> a <https://w3id.org/pmd/tto/StandardForce> ;
    rdfs:label "Force (N)" ;
   

## Step 6: Save

Both outputs are written next to this notebook.

In [10]:
stem = data_file.stem

ttl_path = HERE / f'{stem}.ttl'
flat.serialize(destination=str(ttl_path), format='turtle')
print(f'RDF written to      {ttl_path}')

try:
    parquet_path = HERE / f'{stem}.parquet'
    df.to_parquet(parquet_path, index=False)
    print(f'DataFrame written to {parquet_path}')
except ImportError:
    csv_path = HERE / f'{stem}_data.csv'
    df.to_csv(csv_path, index=False)
    print(f'DataFrame written to {csv_path}  (install pyarrow for Parquet)')

RDF written to      /root/semantic-dataspace/semantic-transformers/docs/example_measurement.ttl
DataFrame written to /root/semantic-dataspace/semantic-transformers/docs/example_measurement.parquet


## Summary

| Step | What happens |
|---|---|
| 0 | Point to your file |
| 1 | Inspect columns and data types |
| 2 | Write a mapping config (ontology class + unit per column) |
| 3 | Run `QuickMapper` to produce the RDF graph |
| 4 | Explore the DataFrame |
| 5 | Inspect the RDF graph |
| 6 | Save graph as `.ttl` and data as `.parquet` |

## When to graduate to a formal schema

The `QuickMapper` output is a good starting point for exploration and
one-off conversions. When the same file type recurs regularly, consider
formalising it:

1. **Add a parser** in `semantic-transformers/parsers/` following the
   [adding-a-parser guide](adding-a-parser.md). The `column_mapping.json`
   is essentially the `columns:` block from your mapping config.
2. **Add a schema** in `semantic-schemas` for the domain, with a full
   OO-LD context and SHACL shapes for validation.
3. **Replace `QuickMapper`** with a `Converter` + your new parser to
   get ontology-aligned, validated RDF.